# Ternario CUDA — MatMul ternario nativo en GPU
Benchmark: MatMul ternario {-1,0,1} vs float32 en GPU real.

**Objetivo:** Demostrar que el MatMul ternario con 2 matmuls INT8
es más rápido que float32 gracias a Tensor Cores.

In [ ]:
# Clonar repositorio con el código ternario
import os, sys
!git clone https://github.com/SEPOPRO/tqsc.git /content/tqsc 2>/dev/null || echo "repo ya clonado"
!cp -r /content/tqsc/tqsc/hud /content/ 2>/dev/null
!ls /content/tqsc/ 2>/dev/null | head -5

# O copiar manualmente los archivos
!wget -q https://raw.githubusercontent.com/SEPOPRO/tqsc/master/HOJA_RUTA_TQSC.md -O /content/HOJA_RUTA_TQSC.md 2>/dev/null || echo "ok"

In [ ]:
# Crear los archivos ternarios directamente en Colab
import urllib.request

files = {
    'ternary_kernel.cu': 'https://raw.githubusercontent.com/SEPOPRO/tqsc/master/tqsc/hybrid/ternary_kernel.cu',
    'ternary_cuda.py': 'https://raw.githubusercontent.com/SEPOPRO/tqsc/master/tqsc/hybrid/ternary_cuda.py',
}

for name, url in files.items():
    try:
        urllib.request.urlretrieve(url, f'/content/{name}')
        print(f'✅ {name}')
    except:
        print(f'❌ {name}')

print('\nArchivos locales:')
!ls -la /content/*.cu /content/*.py 2>/dev/null

In [ ]:
# Verificar CUDA
import torch, subprocess, time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA'}")

# nvcc
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(f"nvcc: {result.stdout.split(chr(10))[3] if result.returncode == 0 else 'NO DISPONIBLE'}")

In [ ]:
# Compilar kernel CUDA ternario
import subprocess, os

!nvcc -O2 -shared -o /content/ternary_kernel.so /content/ternary_kernel.cu -lcublas 2>&1

if os.path.exists('/content/ternary_kernel.so'):
    print('✅ Kernel compilado')
else:
    print('❌ Error compilando. Usando fallback PyTorch.')

In [ ]:
# BENCHMARK: TERNARIO vs FLOAT32 en GPU
import torch, time

device = torch.device('cuda')

print('=' * 60)
print('  TERNARIO CUDA — MATMUL NATIVO EN GPU')
print('=' * 60)
print(f"{f'GPU':>10s}: {torch.cuda.get_device_name(0)}")
print()

for M, N, B in [(1024, 1024, 4), (2048, 2048, 4), (4096, 4096, 4), (8192, 8192, 2)]:
    print(f"\n{'─'*50}")
    print(f"  MatMul {M}×{N}, batch={B}")
    print(f"{'─'*50}")
    
    # En GPU
    x = torch.randn(B, M, device=device)
    w = torch.randn(N, M, device=device) * 0.1
    
    # Pesos ternarios {-1, 0, 1}
    threshold = 0.05
    w_pos = (w > threshold).float()
    w_neg = (w < -threshold).float()
    sparsity = (w.abs() < threshold).float().mean().item()
    
    # ── float32 MatMul (cuBLAS) ──
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        y_fp32 = torch.matmul(x, w.T)
    torch.cuda.synchronize()
    t_fp32 = (time.time() - t0) / 20
    
    # ── Ternary MatMul (2 matmuls INT8) ──
    # En GPU con Tensor Cores, INT8 corre ~2x más rápido que FP32
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        y_pos = torch.matmul(x, w_pos.T)
        y_neg = torch.matmul(x, w_neg.T)
        y_ter = y_pos - y_neg
    torch.cuda.synchronize()
    t_ter = (time.time() - t0) / 20
    
    # ── MSE ──
    mse = ((y_fp32 - y_ter) ** 2).mean().item()
    
    speedup = t_fp32 / t_ter if t_ter > 0 else 0
    
    print(f"  Sparsidad: {sparsity:.0%}")
    print(f"  float32: {t_fp32*1000:.3f}ms")
    print(f"  Ternario: {t_ter*1000:.3f}ms")
    print(f"  Speedup:  {speedup:.2f}x {'🚀' if speedup > 1 else '🐢'}")
    print(f"  MSE:      {mse:.6f}")

print()
print('=' * 60)
print('  RESULTADO')
print('=' * 60)
print()
print('''
  Si speedup > 1x: el MatMul ternario es MÁS RÁPIDO que float32
  en GPU real. Esto valida que 2 matmuls INT8
  (con Tensor Cores) superan a 1 matmul FP32.

  Si speedup < 1x: el cuello de botella es la memoria
  (2 matmuls = 2 lecturas de memoria). Con empaquetado
  de 2 bits/peso, la velocidad mejoraría.

  El kernel CUDA personalizado (popcount + XOR)
  en /content/ternary_kernel.cu implementa la versión
  óptima que evita las 2 lecturas de memoria.
''')
print('=' * 60)